# ASH-KV Validation Suite: Reconstruction & Latency

In [ ]:
import torch
import time
from ashkv.sglang_integration.allocator import SGLangShadowAllocator
from ashkv.sglang_integration.hooks import SGLangHooks
from ashkv.codecs import get_codec_registry

class MockTokenToKVPool:
    def __init__(self, capacity=8000):
        self.capacity = capacity
        self.allocated_tokens = 0
        self.next_idx = 0
        
    def allocate(self, num_tokens):
        if self.allocated_tokens + num_tokens > self.capacity:
            return None
        idx = torch.arange(self.next_idx, self.next_idx + num_tokens, device="cuda")
        self.allocated_tokens += num_tokens
        self.next_idx += num_tokens
        return idx
        
    def free(self, indices):
        self.allocated_tokens -= len(indices)

class MockNode:
    def __init__(self, value):
        self.value = value
        self.kv_indices = value
        self.is_compressed = False

# Setup Environment
pool = MockTokenToKVPool(capacity=8000)
shadow_alloc = SGLangShadowAllocator(max_bytes=100 * 1024 * 1024)
sglang_kv_cache = torch.zeros((32, 8000, 128), dtype=torch.bfloat16, device="cuda")

class MockModelConfig:
    def __init__(self):
        self.num_hidden_layers = 32

hooks = SGLangHooks(MockModelConfig(), shadow_alloc, "int8")


In [ ]:
print("--- TEST 1: RECONSTRUCTION VALIDATION ---")
num_tokens = 4000
original_idx = pool.allocate(num_tokens)
node = MockNode(original_idx)

# Generate known synthetic data and populate the physical cache
synthetic_data = torch.randn((num_tokens, 128), dtype=torch.bfloat16, device="cuda")

# Let's populate layer 0 for the test
sglang_kv_cache[0][original_idx] = synthetic_data

# 1. Compress (Demote)
success = hooks.demote_hook(node, sglang_kv_cache, pool)
assert success, "Demote hook failed"
assert node.is_compressed, "Node not marked compressed"

# 2. Clear Physical VRAM (Ensure no ghost data)
sglang_kv_cache.zero_()

# 3. Decompress (Promote)
hooks.promote_hook(node, sglang_kv_cache, pool)
assert not node.is_compressed, "Node still marked compressed"

# 4. Validate Reconstruction
restored_data = sglang_kv_cache[0][node.value]

is_close = torch.allclose(synthetic_data, restored_data, atol=1e-2)
max_err = torch.max(torch.abs(synthetic_data - restored_data)).item()

print(f"Reconstruction Match (atol=1e-2): {is_close}")
print(f"Max Absolute Error: {max_err:.4f}")
assert is_close, "Reconstruction failed torch.allclose check!"
print("Test 1 Passed.\n")


In [ ]:
print("--- TEST 2: LATENCY PROFILING (MI300X SPECIFIC) ---")

token_sizes = [400, 1000, 4000]
results = []

for size in token_sizes:
    # Warmup node
    idx = pool.allocate(size)
    node = MockNode(idx)
    sglang_kv_cache[0][idx] = torch.randn((size, 128), dtype=torch.bfloat16, device="cuda")
    
    # Measure Demote (Compression)
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    torch.cuda.synchronize()
    start_event.record()
    hooks.demote_hook(node, sglang_kv_cache, pool)
    end_event.record()
    torch.cuda.synchronize()
    
    demote_time_ms = start_event.elapsed_time(end_event)
    
    # Measure Promote (Decompression)
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    torch.cuda.synchronize()
    start_event.record()
    hooks.promote_hook(node, sglang_kv_cache, pool)
    end_event.record()
    torch.cuda.synchronize()
    
    promote_time_ms = start_event.elapsed_time(end_event)
    
    results.append({
        "Tokens": size,
        "Demote (ms)": f"{demote_time_ms:.3f}",
        "Promote (ms)": f"{promote_time_ms:.3f}"
    })
    
    # Cleanup
    pool.free(node.value)

# Output Markdown Table
print("| Tokens | Demote / Compress (ms) | Promote / Decompress (ms) |")
print("|--------|------------------------|---------------------------|")
for r in results:
    print(f"| {r['Tokens']} | {r['Demote (ms)']} | {r['Promote (ms)']} |")

print("\n*Note: Latency numbers are strictly specific to MI300X execution.*")
